# 07_05 — Calibração Venn-Abers — Early-Stage

**Projeto:** `dsc_legal_intelligence`  
**Objetivo:** calibrar as probabilidades do modelo early-stage para o target:

```text
0 = banco ganhou
1 = banco perdeu
```

Este notebook parte dos artefatos criados nas etapas anteriores e usa o holdout temporal apenas para avaliação final. A calibração é necessária porque existe drift temporal relevante na taxa de perda e o score bruto pode ranquear bem sem representar uma probabilidade confiável.

## 1. Imports e configurações

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    precision_score, recall_score, f1_score,
    brier_score_loss, log_loss, classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression

import matplotlib.pyplot as plt
import joblib

RANDOM_STATE = 42
TARGET_COL = "target_banco_ganhou"  # semântica real: 0=banco ganhou | 1=banco perdeu
DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports" / "modelagem_early_stage"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEV_PATH = DATA_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_PATH = DATA_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_PATH = DATA_DIR / "feature_list_early_stage_v1_hist.txt"

print("DEV_PATH:", DEV_PATH)
print("HOLDOUT_PATH:", HOLDOUT_PATH)
print("FEATURE_LIST_PATH:", FEATURE_LIST_PATH)
print("REPORT_DIR:", REPORT_DIR)

## 2. Funções auxiliares

In [ ]:
def save_report(df: pd.DataFrame, filename: str) -> Path:
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path


def read_feature_list(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def infer_feature_types(df: pd.DataFrame, features: list[str]):
    numeric_features, categorical_features, text_features = [], [], []
    text_name_patterns = ["texto", "descricao", "assunto_texto", "classe_texto", "area_texto", "full_text", "text"]
    for c in features:
        if c not in df.columns:
            continue
        if any(p in c.lower() for p in text_name_patterns):
            text_features.append(c)
        elif pd.api.types.is_numeric_dtype(df[c]):
            numeric_features.append(c)
        else:
            categorical_features.append(c)
    return numeric_features, categorical_features, text_features


def make_preprocessor(numeric_features, categorical_features, text_features,
                      tfidf_max_features=1500, tfidf_min_df=20,
                      onehot_min_frequency=20, scale_numeric=False):
    transformers = []
    if numeric_features:
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(steps), numeric_features))
    if categorical_features:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=onehot_min_frequency)),
        ])
        transformers.append(("cat", cat_pipe, categorical_features))
    for txt_col in text_features:
        txt_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="")),
            ("flatten", FunctionTransformer(lambda x: np.asarray(x).ravel(), validate=False)),
            ("tfidf", TfidfVectorizer(max_features=tfidf_max_features, min_df=tfidf_min_df,
                                      ngram_range=(1, 2), strip_accents="unicode", lowercase=True)),
        ])
        transformers.append((f"txt_{txt_col}", txt_pipe, [txt_col]))
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)


def get_positive_proba(estimator, X) -> np.ndarray:
    # Classe positiva = 1 = banco perdeu.
    proba = estimator.predict_proba(X)
    return proba[:, 1] if getattr(proba, "ndim", 1) == 2 else proba


def threshold_metrics(y_true, proba, threshold=0.5) -> dict:
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def best_f1_threshold(y_true, proba) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    if len(thresholds) == 0:
        return {"best_f1_threshold_perda": 0.5, "best_f1_precision_perda": np.nan,
                "best_f1_recall_perda": np.nan, "best_f1_perda": np.nan}
    best_idx = int(np.nanargmax(f1[:-1]))
    return {
        "best_f1_threshold_perda": float(thresholds[best_idx]),
        "best_f1_precision_perda": float(precision[best_idx]),
        "best_f1_recall_perda": float(recall[best_idx]),
        "best_f1_perda": float(f1[best_idx]),
    }


def expected_calibration_error(y_true, proba, n_bins=10) -> float:
    y_true, proba = np.asarray(y_true), np.asarray(proba)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        mask = (proba >= left) & (proba <= right if i == n_bins - 1 else proba < right)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(y_true[mask].mean() - proba[mask].mean())
    return float(ece)


def calibration_report(y_true, proba, label: str, n_bins=10) -> pd.DataFrame:
    frac_pos, mean_pred = calibration_curve(y_true, proba, n_bins=n_bins, strategy="quantile")
    return pd.DataFrame({
        "model": label,
        "bin": np.arange(1, len(frac_pos) + 1),
        "mean_predicted_proba_perda": mean_pred,
        "observed_loss_rate": frac_pos,
        "gap_observed_minus_predicted": frac_pos - mean_pred,
    })


def evaluate_proba(y_true, proba, label: str) -> dict:
    out = {
        "model": label,
        "roc_auc_perda": roc_auc_score(y_true, proba),
        "pr_auc_perda": average_precision_score(y_true, proba),
        "brier_loss": brier_score_loss(y_true, proba),
        "log_loss": log_loss(y_true, np.clip(proba, 1e-6, 1 - 1e-6)),
        "ece_10_bins": expected_calibration_error(y_true, proba, n_bins=10),
        "taxa_perda_real": float(np.mean(y_true)),
        "proba_perda_media": float(np.mean(proba)),
        "proba_perda_p50": float(np.quantile(proba, 0.50)),
        "proba_perda_p90": float(np.quantile(proba, 0.90)),
        "proba_perda_p95": float(np.quantile(proba, 0.95)),
    }
    out.update(threshold_metrics(y_true, proba, threshold=0.5))
    out.update(best_f1_threshold(y_true, proba))
    return out


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({"y_true": y_true, "proba_perda": proba_perda}).sort_values("proba_perda", ascending=False)
    base_loss_rate = df["y_true"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y_true"].mean()
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": top["y_true"].sum() / max(df["y_true"].sum(), 1),
            "lift_perda_at_k": precision_at_k / base_loss_rate if base_loss_rate > 0 else np.nan,
            "taxa_perda_base": base_loss_rate,
        })
    return pd.DataFrame(rows)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({"y_true": y_true, "proba_perda": proba_perda, "valor_ajuizado": valor_ajuizado}).copy()
    df["valor_ajuizado"] = pd.to_numeric(df["valor_ajuizado"], errors="coerce").fillna(0).clip(lower=0)
    df["prioridade_financeira"] = df["proba_perda"] * df["valor_ajuizado"]
    df["valor_perdido_real"] = df["y_true"] * df["valor_ajuizado"]
    df = df.sort_values("prioridade_financeira", ascending=False)
    total_valor, total_perdas = df["valor_ajuizado"].sum(), df["valor_perdido_real"].sum()
    base_loss_rate = df["y_true"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y_true"].mean()
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": top["y_true"].sum() / max(df["y_true"].sum(), 1),
            "lift_perda_at_k": precision_at_k / base_loss_rate if base_loss_rate > 0 else np.nan,
            "taxa_perda_base": base_loss_rate,
            "valor_ajuizado_top": top["valor_ajuizado"].sum(),
            "share_valor_ajuizado_total": top["valor_ajuizado"].sum() / total_valor if total_valor > 0 else np.nan,
            "valor_ajuizado_perdas_top": top["valor_perdido_real"].sum(),
            "share_valor_perdas_total": top["valor_perdido_real"].sum() / total_perdas if total_perdas > 0 else np.nan,
        })
    return pd.DataFrame(rows)

## 3. Carregar DEV, holdout e feature list

In [ ]:
assert DEV_PATH.exists(), f"Arquivo não encontrado: {DEV_PATH}"
assert HOLDOUT_PATH.exists(), f"Arquivo não encontrado: {HOLDOUT_PATH}"
assert FEATURE_LIST_PATH.exists(), f"Arquivo não encontrado: {FEATURE_LIST_PATH}"

df_dev = pd.read_parquet(DEV_PATH)
df_holdout = pd.read_parquet(HOLDOUT_PATH)
feature_list = read_feature_list(FEATURE_LIST_PATH)

blocked = {TARGET_COL, DATE_COL, "Dataencerramento", "Motivo_Encerramento", "Situacao", "desfecho_categoria"}
feature_list = [f for f in feature_list if f in df_dev.columns and f in df_holdout.columns and f not in blocked]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")
df_dev = df_dev.sort_values(DATE_COL).reset_index(drop=True)
df_holdout = df_holdout.sort_values(DATE_COL).reset_index(drop=True)

df_dev[TARGET_COL] = pd.to_numeric(df_dev[TARGET_COL], errors="coerce").astype(int)
df_holdout[TARGET_COL] = pd.to_numeric(df_holdout[TARGET_COL], errors="coerce").astype(int)

summary_load = pd.DataFrame([{
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(feature_list),
    "taxa_perda_dev": df_dev[TARGET_COL].mean(),
    "taxa_perda_holdout": df_holdout[TARGET_COL].mean(),
    "dev_min_date": df_dev[DATE_COL].min(),
    "dev_max_date": df_dev[DATE_COL].max(),
    "holdout_min_date": df_holdout[DATE_COL].min(),
    "holdout_max_date": df_holdout[DATE_COL].max(),
}])

save_report(summary_load, "90_05_load_summary.csv")
summary_load

## 4. Split temporal: treino efetivo vs calibração

Usa o bloco mais recente do DEV para calibrar. Esta escolha simula melhor produção do que split aleatório.

In [ ]:
CALIBRATION_SIZE = 0.20
cut_idx = int(np.floor(len(df_dev) * (1 - CALIBRATION_SIZE)))

df_train_eff = df_dev.iloc[:cut_idx].copy()
df_cal = df_dev.iloc[cut_idx:].copy()

X_train_eff = df_train_eff[feature_list].copy()
y_train_eff = df_train_eff[TARGET_COL].copy()
X_cal = df_cal[feature_list].copy()
y_cal = df_cal[TARGET_COL].copy()
X_holdout = df_holdout[feature_list].copy()
y_holdout = df_holdout[TARGET_COL].copy()

split_cal_summary = pd.DataFrame([
    {"base": "train_eff", "qtd": len(df_train_eff), "inicio": df_train_eff[DATE_COL].min(), "fim": df_train_eff[DATE_COL].max(), "taxa_perda": y_train_eff.mean(), "taxa_ganho": 1-y_train_eff.mean()},
    {"base": "calibration", "qtd": len(df_cal), "inicio": df_cal[DATE_COL].min(), "fim": df_cal[DATE_COL].max(), "taxa_perda": y_cal.mean(), "taxa_ganho": 1-y_cal.mean()},
    {"base": "holdout", "qtd": len(df_holdout), "inicio": df_holdout[DATE_COL].min(), "fim": df_holdout[DATE_COL].max(), "taxa_perda": y_holdout.mean(), "taxa_ganho": 1-y_holdout.mean()},
])

save_report(split_cal_summary, "91_05_temporal_calibration_split.csv")
split_cal_summary

## 5. Criar modelo base para calibração

Por padrão, usa Random Forest + OneHot + TF-IDF, que foi competitivo nas etapas anteriores. O foco deste notebook é calibrar probabilidades, não substituir a etapa de tuning.

In [ ]:
numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)
print("numeric_features:", len(numeric_features))
print("categorical_features:", len(categorical_features))
print("text_features:", len(text_features), text_features[:5])

preprocessor = make_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    text_features=text_features,
    tfidf_max_features=1500,
    tfidf_min_df=20,
    onehot_min_frequency=20,
    scale_numeric=False,
)

base_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=18,
    min_samples_split=30,
    min_samples_leaf=15,
    max_features="sqrt",
    max_samples=0.8,
    class_weight="balanced",
    criterion="entropy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

pipeline_raw = Pipeline([("preprocess", preprocessor), ("model", base_model)])
pipeline_raw

## 6. Treinar modelo bruto no treino efetivo

In [ ]:
pipeline_raw.fit(X_train_eff, y_train_eff)

proba_cal_raw = get_positive_proba(pipeline_raw, X_cal)
proba_holdout_raw = get_positive_proba(pipeline_raw, X_holdout)

raw_metrics_cal = evaluate_proba(y_cal, proba_cal_raw, "raw_calibration_block")
raw_metrics_holdout = evaluate_proba(y_holdout, proba_holdout_raw, "raw_holdout")
raw_eval = pd.DataFrame([raw_metrics_cal, raw_metrics_holdout])
save_report(raw_eval, "92_05_raw_model_cal_holdout_metrics.csv")
raw_eval

## 7. Calibração Venn-Abers

O bloco abaixo tenta usar `venn-abers`. Se a lib não estiver disponível, usa isotonic regression como fallback diagnóstico e marca isso claramente nos outputs.

In [ ]:
def fit_predict_venn_abers_or_fallback(y_cal, proba_cal_raw, proba_holdout_raw, fallback_to_isotonic=True):
    y_cal_arr = np.asarray(y_cal).astype(int)
    p_cal = np.asarray(proba_cal_raw).astype(float)
    p_hold = np.asarray(proba_holdout_raw).astype(float)

    try:
        from venn_abers import VennAbersCalibrator
        try:
            va = VennAbersCalibrator()
            va.fit(p_cal.reshape(-1, 1), y_cal_arr)
            p_calibrated = va.predict_proba(p_hold.reshape(-1, 1))
            if isinstance(p_calibrated, tuple):
                p0, p1 = p_calibrated
                p_out = (np.asarray(p0) + np.asarray(p1)) / 2
            else:
                p_calibrated = np.asarray(p_calibrated)
                p_out = p_calibrated[:, 1] if p_calibrated.ndim == 2 else p_calibrated
            return np.clip(p_out, 0, 1), "venn_abers"
        except Exception as e1:
            try:
                va = VennAbersCalibrator(inductive=True)
                va.fit(p_cal.reshape(-1, 1), y_cal_arr)
                p_calibrated = np.asarray(va.predict_proba(p_hold.reshape(-1, 1)))
                p_out = p_calibrated[:, 1] if p_calibrated.ndim == 2 else p_calibrated
                return np.clip(p_out, 0, 1), "venn_abers"
            except Exception as e2:
                print("VennAbersCalibrator encontrado, mas a API falhou.")
                print("Erro 1:", repr(e1))
                print("Erro 2:", repr(e2))
                if not fallback_to_isotonic:
                    raise
    except Exception as e:
        print("Biblioteca venn_abers não disponível ou import falhou:", repr(e))
        if not fallback_to_isotonic:
            raise

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_cal, y_cal_arr)
    p_out = iso.predict(p_hold)
    return np.clip(p_out, 0, 1), "fallback_isotonic_not_venn_abers"

proba_holdout_calibrated, calibration_method = fit_predict_venn_abers_or_fallback(
    y_cal=y_cal,
    proba_cal_raw=proba_cal_raw,
    proba_holdout_raw=proba_holdout_raw,
    fallback_to_isotonic=True,
)
print("Método de calibração usado:", calibration_method)

## 8. Avaliar probabilidade calibrada no holdout

In [ ]:
calibrated_metrics_holdout = evaluate_proba(y_holdout, proba_holdout_calibrated, f"holdout_calibrated_{calibration_method}")
comparison_calibration = pd.DataFrame([raw_metrics_holdout, calibrated_metrics_holdout])

metric_delta_cols = ["roc_auc_perda", "pr_auc_perda", "brier_loss", "log_loss", "ece_10_bins", "proba_perda_media", "best_f1_perda", "precision_perda", "recall_perda", "f1_perda"]
raw_row, cal_row = comparison_calibration.iloc[0], comparison_calibration.iloc[1]
comparison_deltas = pd.DataFrame([{f"delta_{c}_calibrated_minus_raw": cal_row[c] - raw_row[c] for c in metric_delta_cols}])

save_report(comparison_calibration, "93_05_calibration_metrics_comparison.csv")
save_report(comparison_deltas, "94_05_calibration_metrics_deltas.csv")
comparison_calibration

## 9. Curvas de calibração

In [ ]:
cal_report_raw = calibration_report(y_holdout, proba_holdout_raw, "raw_holdout", n_bins=10)
cal_report_cal = calibration_report(y_holdout, proba_holdout_calibrated, f"calibrated_{calibration_method}", n_bins=10)
calibration_bins = pd.concat([cal_report_raw, cal_report_cal], ignore_index=True)
save_report(calibration_bins, "95_05_calibration_bins.csv")

plt.figure(figsize=(8, 6))
for label, grp in calibration_bins.groupby("model"):
    plt.plot(grp["mean_predicted_proba_perda"], grp["observed_loss_rate"], marker="o", label=label)
plt.plot([0, 1], [0, 1], linestyle="--", label="perfeitamente calibrado")
plt.xlabel("Probabilidade média prevista de perda")
plt.ylabel("Taxa observada de perda")
plt.title("Curva de calibração — Holdout")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / "95_05_calibration_curve_holdout.png", dpi=160)
plt.show()

calibration_bins

## 10. Curva Precision-Recall antes/depois

In [ ]:
def plot_pr_curve(y_true, proba_dict):
    plt.figure(figsize=(8, 6))
    base = np.mean(y_true)
    plt.axhline(base, linestyle="--", label=f"baseline taxa perda={base:.3f}")
    for label, proba in proba_dict.items():
        p, r, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        plt.plot(r, p, label=f"{label} | AP={ap:.3f}")
    plt.xlabel("Recall perda")
    plt.ylabel("Precision perda")
    plt.title("Precision-Recall — classe 1 = banco perdeu")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "96_05_precision_recall_raw_vs_calibrated.png", dpi=160)
    plt.show()

plot_pr_curve(y_holdout, {"raw": proba_holdout_raw, calibration_method: proba_holdout_calibrated})

## 11. Threshold operacional pós-calibração

In [ ]:
threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for t in threshold_grid:
    r = threshold_metrics(y_holdout, proba_holdout_calibrated, threshold=float(t))
    r["model"] = f"calibrated_{calibration_method}"
    rows.append(r)
threshold_table = pd.DataFrame(rows)
save_report(threshold_table, "97_05_threshold_table_calibrated.csv")
threshold_table.sort_values("f1_perda", ascending=False).head(10)

## 12. Top-k de risco de perda — raw vs calibrado

In [ ]:
topk_raw = topk_risco_perda_metrics(y_holdout.values, proba_holdout_raw)
topk_raw["model"] = "raw"
topk_cal = topk_risco_perda_metrics(y_holdout.values, proba_holdout_calibrated)
topk_cal["model"] = f"calibrated_{calibration_method}"
topk_compare = pd.concat([topk_raw, topk_cal], ignore_index=True)
save_report(topk_compare, "98_05_topk_risco_perda_raw_vs_calibrated.csv")
topk_compare

## 13. Top-k de prioridade financeira — raw vs calibrado

In [ ]:
if VALUE_COL in df_holdout.columns:
    topk_fin_raw = topk_prioridade_financeira_metrics(y_holdout.values, proba_holdout_raw, df_holdout[VALUE_COL].values)
    topk_fin_raw["model"] = "raw"
    topk_fin_cal = topk_prioridade_financeira_metrics(y_holdout.values, proba_holdout_calibrated, df_holdout[VALUE_COL].values)
    topk_fin_cal["model"] = f"calibrated_{calibration_method}"
    topk_fin_compare = pd.concat([topk_fin_raw, topk_fin_cal], ignore_index=True)
    save_report(topk_fin_compare, "99_05_topk_prioridade_financeira_raw_vs_calibrated.csv")
    display(topk_fin_compare)
else:
    print(f"Coluna {VALUE_COL} não encontrada no holdout. Pulando análise financeira.")

## 14. Ranking calibrado para consumo jurídico

In [ ]:
ranking_cols = []
for c in ["Numerodistribuicao", "Identificador", DATE_COL, VALUE_COL, "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca"]:
    if c in df_holdout.columns and c not in ranking_cols:
        ranking_cols.append(c)

ranking_holdout = df_holdout[ranking_cols].copy() if ranking_cols else pd.DataFrame(index=df_holdout.index)
ranking_holdout[TARGET_COL] = y_holdout.values
ranking_holdout["target_semantica"] = np.where(ranking_holdout[TARGET_COL] == 1, "banco_perdeu", "banco_ganhou")
ranking_holdout["proba_perda_raw"] = proba_holdout_raw
ranking_holdout["proba_perda_calibrada"] = proba_holdout_calibrated
ranking_holdout["delta_calibrada_minus_raw"] = ranking_holdout["proba_perda_calibrada"] - ranking_holdout["proba_perda_raw"]
ranking_holdout["ranking_risco_perda"] = ranking_holdout["proba_perda_calibrada"].rank(ascending=False, method="first").astype(int)

if VALUE_COL in ranking_holdout.columns:
    ranking_holdout["prioridade_financeira_calibrada"] = pd.to_numeric(ranking_holdout[VALUE_COL], errors="coerce").fillna(0).clip(lower=0) * ranking_holdout["proba_perda_calibrada"]
    ranking_holdout["ranking_prioridade_financeira"] = ranking_holdout["prioridade_financeira_calibrada"].rank(ascending=False, method="first").astype(int)
else:
    ranking_holdout["prioridade_financeira_calibrada"] = np.nan
    ranking_holdout["ranking_prioridade_financeira"] = np.nan

ranking_holdout = ranking_holdout.sort_values("ranking_risco_perda")
ranking_path = DATA_DIR / "ranking_holdout_risco_perda_calibrado_07_05.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)
print("Ranking salvo em:", ranking_path)
save_report(ranking_holdout.head(1000), "100_05_ranking_holdout_top1000_calibrado.csv")
ranking_holdout.head(20)

## 15. Exportar artefatos

In [ ]:
artifact = {
    "pipeline_raw": pipeline_raw,
    "calibration_method": calibration_method,
    "feature_list": feature_list,
    "target_col": TARGET_COL,
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "date_col": DATE_COL,
    "value_col": VALUE_COL,
    "calibration_size": CALIBRATION_SIZE,
    "train_eff_start": str(df_train_eff[DATE_COL].min()),
    "train_eff_end": str(df_train_eff[DATE_COL].max()),
    "calibration_start": str(df_cal[DATE_COL].min()),
    "calibration_end": str(df_cal[DATE_COL].max()),
}
model_path = MODEL_DIR / "07_05_pipeline_raw_for_venn_abers_calibration.joblib"
joblib.dump(artifact, model_path)
print("Artefato salvo em:", model_path)

prob_table = pd.DataFrame({"y_holdout": y_holdout.values, "proba_perda_raw": proba_holdout_raw, "proba_perda_calibrada": proba_holdout_calibrated})
prob_path = DATA_DIR / "07_05_holdout_probabilidades_raw_calibradas.parquet"
prob_table.to_parquet(prob_path, index=False)
print("Probabilidades salvas em:", prob_path)

## 16. Summary final

In [ ]:
summary_step_05 = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "calibration_method": calibration_method,
    "qtd_dev": len(df_dev),
    "qtd_train_eff": len(df_train_eff),
    "qtd_calibration": len(df_cal),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(feature_list),
    "taxa_perda_calibration": y_cal.mean(),
    "taxa_perda_holdout": y_holdout.mean(),
    "raw_holdout_pr_auc_perda": raw_metrics_holdout["pr_auc_perda"],
    "cal_holdout_pr_auc_perda": calibrated_metrics_holdout["pr_auc_perda"],
    "raw_holdout_roc_auc_perda": raw_metrics_holdout["roc_auc_perda"],
    "cal_holdout_roc_auc_perda": calibrated_metrics_holdout["roc_auc_perda"],
    "raw_brier_loss": raw_metrics_holdout["brier_loss"],
    "cal_brier_loss": calibrated_metrics_holdout["brier_loss"],
    "raw_ece_10_bins": raw_metrics_holdout["ece_10_bins"],
    "cal_ece_10_bins": calibrated_metrics_holdout["ece_10_bins"],
    "ranking_calibrado_path": str(ranking_path),
    "model_artifact_path": str(model_path),
    "probabilities_path": str(prob_path),
}])
save_report(summary_step_05, "101_05_summary_calibracao_venn_abers.csv")
summary_step_05.T

## O que enviar para a próxima iteração

Depois de rodar este notebook, envie:

1. `summary_step_05.T`
2. `comparison_calibration`
3. `comparison_deltas`
4. `topk_compare`
5. `topk_fin_compare`, caso a coluna de valor exista
6. a curva de calibração `95_05_calibration_curve_holdout.png`

Critério de leitura:

- Se `cal_brier_loss` e `cal_ece_10_bins` melhorarem, a calibração foi útil mesmo que PR-AUC/ROC-AUC fiquem semelhantes.
- Se a probabilidade média calibrada ficar mais próxima da taxa real de perda do holdout, o score calibrado fica mais defensável para threshold e priorização.
- Se a calibração piorar muito ranking/top-k, mantenha score bruto para ranking e use score calibrado apenas para comunicação de probabilidade.